# Nurse Navigation - Behavioral Health Across All Extracts

Runs the behavioral health note read across the call extracts named in SOURCE_FILES as one combined population. The extracts are listed explicitly rather than discovered, so a run covers a known set of files. Any other extract sitting in the data folder is reported and left out.

The extracts cover overlapping date ranges, so the same call appears in more than one file. Calls are deduplicated before anything is counted. Every count in this notebook is a count of distinct calls.

Notes are scored by the text of the note rather than by row position, so a call that appears in two extracts is read once, and a later extract only pays for notes that have not been read before.


## 1. Setup

In [ ]:
import os, re, json, glob, time, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from concurrent.futures import ThreadPoolExecutor, as_completed
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200); pd.set_option("display.max_colwidth", 300)
plt.rcParams.update({"figure.figsize":(11,5),"figure.dpi":110,"axes.grid":True,"grid.alpha":0.25,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,"axes.titlesize":13,"axes.titleweight":"bold"})
TEAL, NAVY, CORAL, GOLD, GREY = "#028090","#0B2545","#D1495B","#E0A500","#8FA0A6"

DATA_DIR   = "/Workspace/Users/josh.smitherman@gmr.net/nurse_nav/data"
OUT_DIR    = "/Workspace/Users/josh.smitherman@gmr.net/nurse_nav/results"
PROMPT_DIR = "/Workspace/Users/josh.smitherman@gmr.net/nurse_nav/prompts"

SOURCE_FILES = ["data_2024.xlsx",
                "data_2025.xlsx",
                "data_jan_2026-jul2026.xlsx",
                "data_april2026-aug2026.xlsx",
                "data_aug_2026-oct2026.xlsx"]
PREFER = "newest"
LEGACY_SCORES = "note_scores.parquet"
LEGACY_SOURCE = "data_april2026-aug2026.xlsx"
SCORE_FILE    = "note_scores_combined.parquet"

LLM_MODEL        = "databricks-gpt-oss-120b"
MAX_WORKERS      = 16
CHECKPOINT_EVERY = 2000
MIN_NOTE_CHARS   = 20
RESCORE_ALL      = False

os.makedirs(OUT_DIR, exist_ok=True)
SCORE_PATH = os.path.join(OUT_DIR, SCORE_FILE)
RUN_ID = pd.Timestamp.now().strftime("%Y%m%d_%H%M")
RESULTS = {}
def keep(d, n): RESULTS[n] = d.copy(); return d
def show(d):
    o = d.copy()
    for c in o.columns:
        if o[c].dtype == object: o[c] = o[c].astype(str)
    display(o)
print("run:", RUN_ID)

## 2. Load the prompt

In [ ]:
with open(os.path.join(PROMPT_DIR, "bh_system_prompt.txt")) as fh:
    BH_SYSTEM_PROMPT = fh.read()
PROMPT_FINGERPRINT = hashlib.sha256(BH_SYSTEM_PROMPT.encode("utf-8")).hexdigest()[:12]
print(f"Prompt fingerprint: {PROMPT_FINGERPRINT}  ({len(BH_SYSTEM_PROMPT):,} characters)")
print("A change to the prompt changes the fingerprint and figures are no longer comparable with an earlier run.")

## 3. Load every extract

Each file is loaded and tagged with the file it came from. Column names can differ between extracts, so each field is resolved per file and any field missing from a file is reported rather than filled silently.

In [ ]:
def find_col(cols, exact, parts):
    low = {str(c).strip().lower(): c for c in cols}
    for e in exact:
        if e in low: return low[e]
    for c_low, c in low.items():
        if all(p in c_low for p in parts): return c
    return None

FIELDS = {
 "call_id":  (["work_set_id","session_id","call id","incident id"], ["work","set","id"]),
 "note":     (["notes","nurse notes","call notes"], ["note"]),
 "cause":    (["cause","chief complaint"], ["cause"]),
 "market":   (["market","agency","customer"], ["market"]),
 "date":     (["call date","transaction_create_date","date_time_central","date"], ["date"]),
 "nmtara":   (["nmtara","nmtara breakout"], ["nmtara"]),
 "dispo":    (["disposition","final disposition"], ["disposition"])}

paths = [os.path.join(DATA_DIR, f) for f in SOURCE_FILES]
absent = [os.path.basename(p) for p in paths if not os.path.exists(p)]
if absent:
    raise FileNotFoundError("named in SOURCE_FILES but not present in the data folder: " + ", ".join(absent))
print(f"{len(paths)} extracts in scope")
for f in SOURCE_FILES: print("  ", f)

present = {os.path.basename(p) for p in glob.glob(os.path.join(DATA_DIR, "data_*.xlsx"))}
unlisted = sorted(present - set(SOURCE_FILES))
if unlisted:
    print("In the data folder and not listed in SOURCE_FILES:")
    for f in unlisted: print("  ", f)
    print("Add it to SOURCE_FILES to include it.")

frames, resolution = [], []
for p in paths:
    name = os.path.basename(p)
    raw = pd.read_excel(p)
    raw.columns = [str(c).strip() for c in raw.columns]
    got = {k: find_col(raw.columns, ex, pt) for k, (ex, pt) in FIELDS.items()}
    out = pd.DataFrame(index=raw.index)
    for k, c in got.items():
        out[k] = raw[c] if c else np.nan
    out["source_file"] = name
    frames.append(out)
    row = {"Extract": name, "Rows": len(raw)}
    row.update({f"{k} column": (got[k] if got[k] else "not found") for k in FIELDS})
    resolution.append(row)

res = pd.DataFrame(resolution)
keep(res, "source_columns")
show(res)
allcalls = pd.concat(frames, ignore_index=True)
print(f"rows loaded across all extracts: {len(allcalls):,}")
gaps = [f"{r['Extract']}: {k}" for r in resolution for k in FIELDS if r[f"{k} column"] == "not found"]
if gaps:
    print("Fields not found:")
    for g in gaps: print("  ", g)

## 4. Coverage and overlap between extracts

The extracts cover overlapping periods. This shows the date range each one carries and how much each pair has in common, so the amount of duplication being removed is visible before it is removed.

In [ ]:
allcalls["date"] = pd.to_datetime(allcalls["date"], errors="coerce")
cov = allcalls.groupby("source_file").agg(
    Rows=("date","size"),
    **{"Earliest call":("date","min"), "Latest call":("date","max"), "Rows with no date":("date", lambda s: int(s.isna().sum()))})
cov = cov.reset_index().rename(columns={"source_file":"Extract"})
cov["Earliest call"] = cov["Earliest call"].dt.date.astype(str)
cov["Latest call"]   = cov["Latest call"].dt.date.astype(str)
keep(cov, "extract_coverage")
show(cov)

m = allcalls.dropna(subset=["date"]).copy()
m["month"] = m["date"].dt.to_period("M").astype(str)
bymonth = pd.crosstab(m["month"], m["source_file"])
overlap_months = bymonth.index[(bymonth > 0).sum(axis=1) > 1].tolist()
print(f"Months carried by more than one extract: {len(overlap_months)}")
if overlap_months:
    print("  ", ", ".join(overlap_months))
    dup_rows = int(bymonth.loc[overlap_months].sum().sum() - bymonth.loc[overlap_months].max(axis=1).sum())
    print(f"Rows sitting in a month that more than one extract covers, above the largest single extract: {dup_rows:,}")

fig, ax = plt.subplots(figsize=(13,4.5))
bottom = np.zeros(len(bymonth))
palette = [NAVY, TEAL, GREY, GOLD, CORAL, "#5B8C85", "#8C5B7A"]
for j, c in enumerate(bymonth.columns):
    ax.bar(range(len(bymonth)), bymonth[c], bottom=bottom, color=palette[j % len(palette)], label=str(c))
    bottom = bottom + bymonth[c].values
ax.set_xticks(range(len(bymonth))); ax.set_xticklabels(bymonth.index, rotation=60, fontsize=8)
ax.legend(fontsize=8, ncol=2); ax.set_title("Rows per month by extract, before deduplication"); ax.set_ylabel("rows")
plt.tight_layout(); plt.show()

## 5. Deduplicate to distinct calls

A call identifier is used where every extract carries one. Where it does not, a call is identified by the combination of its note text, date, market and cause, which is what distinguishes one call from another in these extracts.

Where the same call appears in more than one extract, the row from the extract whose calls run latest is kept, so a later pull takes precedence over an earlier one carrying the same call. Setting PREFER to anything other than newest falls back to file name order.

In [ ]:
def norm_text(v):
    return re.sub(r"\s+", " ", str(v)).strip().lower()

has_id = allcalls["call_id"].notna().all() and allcalls["call_id"].astype(str).str.strip().ne("").all()
if has_id:
    allcalls["dedupe_key"] = allcalls["call_id"].astype(str).str.strip()
    key_basis = "call identifier carried by every extract"
else:
    parts = (allcalls["note"].apply(norm_text) + "|" +
             allcalls["date"].astype(str) + "|" +
             allcalls["market"].apply(norm_text) + "|" +
             allcalls["cause"].apply(norm_text))
    allcalls["dedupe_key"] = parts.apply(lambda t: hashlib.sha256(t.encode("utf-8")).hexdigest()[:20])
    key_basis = "note text, date, market and cause combined"

ends = allcalls.groupby("source_file")["date"].max().sort_values(ascending=False)
rank = {f: i for i, f in enumerate(ends.index)} if PREFER == "newest" else {f: i for i, f in enumerate(sorted(allcalls["source_file"].unique()))}
allcalls["_rank"] = allcalls["source_file"].map(rank)
before = len(allcalls)
df = allcalls.sort_values(["_rank","source_file"]).drop_duplicates("dedupe_key", keep="first").drop(columns=["_rank"]).reset_index(drop=True)
removed = before - len(df)
order = pd.DataFrame({"Extract": list(ends.index), "Latest call in extract": [str(pd.Timestamp(d).date()) for d in ends.values],
                      "Precedence on a duplicate": [f"{i+1}" for i in range(len(ends))]})
keep(order, "extract_precedence")
show(order)
print(f"Where a call appears in more than one extract, the row from {ends.index[0]} is kept.")

dedupe = pd.DataFrame({
    "Measure": ["Rows across all extracts", "Duplicate rows removed", "Distinct calls",
                "Basis for identifying a call", "Extract kept on a duplicate"],
    "Value": [f"{before:,}", f"{removed:,} ({removed/before*100:.1f}%)", f"{len(df):,}", key_basis,
              f"the one whose calls run latest ({ends.index[0]})" if PREFER == "newest" else "the first by file name"]})
keep(dedupe, "deduplication")
show(dedupe)

contrib = df["source_file"].value_counts().rename("Distinct calls first seen in this extract").to_frame()
contrib["Share of distinct calls"] = (contrib["Distinct calls first seen in this extract"]/len(df)*100).round(1)
contrib.index.name = "Extract"
keep(contrib.reset_index(), "extract_contribution")
show(contrib)
print(f"Combined population: {len(df):,} distinct calls from {df['date'].min().date()} to {df['date'].max().date()}.")

## 6. Score every note

Notes are keyed by their text. A note already read in an earlier run is not read again, whichever extract it arrived in. Progress is saved so a stopped run resumes where it left off, and re-running retries only the notes that could not be read.

In [ ]:
from openai import OpenAI
DATABRICKS_TOKEN = (dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
                    if "dbutils" in dir() else os.environ.get("DATABRICKS_TOKEN",""))
client = OpenAI(api_key=DATABRICKS_TOKEN, base_url="https://adb-2790612761746757.17.azuredatabricks.net/serving-endpoints")

def llm_call(system, user, max_tokens=700):
    r = client.chat.completions.create(model=LLM_MODEL,
        messages=[{"role":"system","content":system},{"role":"user","content":user}], temperature=0.0, max_tokens=max_tokens)
    c = r.choices[0].message.content
    if isinstance(c, list):
        for it in c:
            if isinstance(it, dict) and it.get("type") == "text": return it.get("text","")
        return json.dumps(c)
    return c

def parse_bh(raw):
    try: o = json.loads(raw)
    except Exception: return None
    return o if "behavioral_health" in o else None

SCORE_COLS = ["note_key","bh_llm","role","category","quote","ok"]

def score_one(key, note):
    delay = 1.0
    for _ in range(4):
        try:
            o = parse_bh(llm_call(BH_SYSTEM_PROMPT, note))
            if o is not None:
                bh = o.get("behavioral_health", {}) or {}
                ev = o.get("evidence") or []
                return {"note_key": key, "bh_llm": bool(bh.get("present", False)),
                        "role": str(bh.get("role","none")), "category": str(bh.get("category","none")),
                        "quote": (ev[0].get("quote","") if ev else "")[:200], "ok": True}
        except Exception:
            pass
        time.sleep(delay); delay *= 2
    return {"note_key": key, "bh_llm": False, "role":"none", "category":"none", "quote":"", "ok": False}

df["note_text"] = df["note"].fillna("").astype(str)
df["note_key"]  = df["note_text"].apply(lambda t: hashlib.sha256(norm_text(t).encode("utf-8")).hexdigest()[:20])
eligible = df[df["note_text"].str.len() >= MIN_NOTE_CHARS]
print(f"Distinct calls          : {len(df):,}")
print(f"Notes long enough to read: {len(eligible):,}")
print(f"Distinct note texts      : {eligible['note_key'].nunique():,}")

### Carry over notes already read

Scores from the earlier single-extract run were keyed by row position. Rejoining them to the extract they came from recovers the note each score belongs to, so those notes are not paid for a second time.

In [ ]:
prior = pd.DataFrame(columns=SCORE_COLS)
if os.path.exists(SCORE_PATH) and not RESCORE_ALL:
    prior = pd.read_parquet(SCORE_PATH)
    print(f"{len(prior):,} scores in the combined checkpoint")

legacy_path = os.path.join(OUT_DIR, LEGACY_SCORES)
legacy_src  = os.path.join(DATA_DIR, LEGACY_SOURCE)
if os.path.exists(legacy_path) and os.path.exists(legacy_src) and not RESCORE_ALL:
    old = pd.read_parquet(legacy_path)
    base = pd.read_excel(legacy_src)
    base.columns = [str(c).strip() for c in base.columns]
    ncol = find_col(base.columns, *FIELDS["note"])
    if ncol and "call_key" in old.columns:
        bridge = pd.DataFrame({"call_key": base.index.astype(str),
                               "note_key": base[ncol].fillna("").astype(str).apply(
                                   lambda t: hashlib.sha256(norm_text(t).encode("utf-8")).hexdigest()[:20])})
        carried = old.merge(bridge, on="call_key", how="inner")
        carried = carried[carried["ok"]][SCORE_COLS].drop_duplicates("note_key", keep="last")
        before_n = prior["note_key"].nunique() if len(prior) else 0
        prior = pd.concat([prior, carried], ignore_index=True).drop_duplicates("note_key", keep="first")
        print(f"carried {len(carried):,} scores over from the earlier run; checkpoint now holds {prior['note_key'].nunique():,}")
    else:
        print("earlier scores found but could not be rejoined to their extract; they will not be carried over")

done = set(prior.loc[prior["ok"], "note_key"]) if len(prior) else set()
todo_keys = sorted(set(eligible["note_key"]) - done)
print(f"Notes already read : {len(done):,}")
print(f"Notes to read now  : {len(todo_keys):,}")
if todo_keys:
    print(f"At {MAX_WORKERS} at a time this takes roughly {len(todo_keys)/MAX_WORKERS*1.5/3600:.1f} hours.")

In [ ]:
if todo_keys:
    first_text = eligible.drop_duplicates("note_key").set_index("note_key")["note_text"]
    work = [(k, first_text[k]) for k in todo_keys]
    acc = prior.to_dict("records") if len(prior) else []
    t0, n = time.time(), 0
    def save():
        pd.DataFrame(acc).drop_duplicates("note_key", keep="last")[SCORE_COLS].to_parquet(SCORE_PATH, index=False)
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        for fut in as_completed([ex.submit(score_one, k, t) for k, t in work]):
            acc.append(fut.result()); n += 1
            if n % CHECKPOINT_EVERY == 0:
                save()
                rate = n / max(time.time()-t0, 1e-9)
                print(f"  {n:,} of {len(work):,}   {rate:.1f} notes/sec   about {(len(work)-n)/max(rate,1e-9)/60:.0f} min left")
    save()
    print(f"Finished in {(time.time()-t0)/60:.1f} minutes.")
elif len(prior):
    prior.drop_duplicates("note_key", keep="last")[SCORE_COLS].to_parquet(SCORE_PATH, index=False)

scores = pd.read_parquet(SCORE_PATH) if os.path.exists(SCORE_PATH) else pd.DataFrame(columns=SCORE_COLS)
ok_rows = scores[scores["ok"]].drop_duplicates("note_key", keep="last").set_index("note_key") if len(scores) else pd.DataFrame()
n_fail = int((~scores["ok"]).sum()) if len(scores) else 0

df["bh_llm"]   = df["note_key"].map(ok_rows["bh_llm"]).fillna(False).astype(bool) if len(ok_rows) else False
df["bh_role"]  = df["note_key"].map(ok_rows["role"]).fillna("none") if len(ok_rows) else "none"
df["bh_cat"]   = df["note_key"].map(ok_rows["category"]).fillna("none") if len(ok_rows) else "none"
df["bh_quote"] = df["note_key"].map(ok_rows["quote"]).fillna("") if len(ok_rows) else ""
df["bh_read"]  = df["note_key"].isin(ok_rows.index) & (df["note_text"].str.len() >= MIN_NOTE_CHARS) if len(ok_rows) else False

print(f"Calls with a note read : {int(df['bh_read'].sum()):,} of {len(df):,}")
print(f"Notes that could not be read : {n_fail:,}")
print(f"Behavioral health anywhere in the note : {int(df['bh_llm'].sum()):,}")
print(f"Behavioral health as the reason for the call : {int((df['bh_llm'] & (df['bh_role']=='primary')).sum()):,}")

## 7. Behavioral health volume across the combined population

In [ ]:
n_all  = len(df)
n_read = int(df["bh_read"].sum())
n_any  = int(df["bh_llm"].sum())
n_prim = int((df["bh_llm"] & (df["bh_role"]=="primary")).sum())
n_sec  = n_any - n_prim
counts = pd.DataFrame({
    "Measure": ["Distinct calls", "Notes read",
                "Behavioral health anywhere in the note",
                "Behavioral health as the reason for the call",
                "Behavioral health as a secondary factor"],
    "Calls": [n_all, n_read, n_any, n_prim, n_sec],
    "Share of distinct calls": [100.0, round(n_read/n_all*100,1), round(n_any/n_all*100,1),
                                round(n_prim/n_all*100,1), round(n_sec/n_all*100,1)]})
keep(counts, "bh_counts")
show(counts)

o = counts.iloc[[2,3,4]]
fig, ax = plt.subplots(figsize=(10,3.2))
ax.barh(range(len(o)), o["Calls"], color=[GREY, TEAL, NAVY])
ax.set_yticks(range(len(o))); ax.set_yticklabels(o["Measure"], fontsize=10)
ax.set_title("Behavioral health across the combined population")
for i, v in enumerate(o["Calls"]):
    ax.annotate(f"{v:,}", (v, i), xytext=(4,0), textcoords="offset points", va="center", fontsize=9)
plt.tight_layout(); plt.show()

## 8. Agreement with the coded field

Calls where a behavioral health code was entered, against what the note records.

In [ ]:
PATHWAY_BH_TERMS = ["behavior","mental","psych","suicid","overdose","alcohol","substance","domestic abuse","child abuse"]
def norm_code(v):
    t = str(v).strip()
    if not t or t.lower() in ("nan","none"): return ""
    m = re.search(r"(bmh)\s*[\.\-]?\s*(\d{1,2})", t, flags=re.I)
    if m: return f"BMH.{int(m.group(2)):02d}"
    low = t.lower()
    if any(k in low for k in PATHWAY_BH_TERMS): return "Behavioral, no code"
    return ""

agree = pd.DataFrame()
df["coded_value"] = df["cause"].fillna("").astype(str).apply(norm_code)
df["coded_bh"] = df["coded_value"].ne("")
r = df[df["bh_read"]]
coded = r[r["coded_bh"]]
if len(coded):
    c_any  = int(coded["bh_llm"].sum())
    c_prim = int((coded["bh_llm"] & (coded["bh_role"]=="primary")).sum())
    prim   = r[r["bh_llm"] & (r["bh_role"]=="primary")]
    p_coded= int(prim["coded_bh"].sum())
    agree = pd.DataFrame({
        "Measure": ["Calls carrying a behavioral health code",
                    "Of those, the note confirms behavioral health",
                    "Of those, the note agrees it was the reason for the call",
                    "Of those, the note does not find behavioral health",
                    "Calls the note says were about behavioral health",
                    "Of those, carrying a behavioral health code",
                    "Of those, carrying no code"],
        "Calls": [len(coded), c_any, c_prim, len(coded)-c_any, len(prim), p_coded, len(prim)-p_coded],
        "Share": [100.0, round(c_any/len(coded)*100,1), round(c_prim/len(coded)*100,1),
                  round((len(coded)-c_any)/len(coded)*100,1), 100.0,
                  round(p_coded/max(len(prim),1)*100,1), round((len(prim)-p_coded)/max(len(prim),1)*100,1)]})
    keep(agree, "agreement")
    show(agree)
    print(f"The note confirms behavioral health on {c_any/len(coded)*100:.1f}% of calls carrying a code.")

    cells = pd.DataFrame({"Outcome": ["Code entered, note does not describe behavioral health",
                                      "No code entered, note does describe behavioral health"],
                          "Calls": [len(coded)-c_any, int((~r["coded_bh"] & r["bh_llm"]).sum())]})
    fig, ax = plt.subplots(figsize=(10,2.6))
    ax.barh(range(len(cells)), cells["Calls"], color=[CORAL, GOLD])
    ax.set_yticks(range(len(cells))); ax.set_yticklabels(cells["Outcome"], fontsize=9.5)
    ax.set_title("Where the coded field and the note disagree")
    for i, v in enumerate(cells["Calls"]):
        ax.annotate(f"{v:,}", (v, i), xytext=(4,0), textcoords="offset points", va="center", fontsize=9)
    plt.tight_layout(); plt.show()

## 9. Categories and markets

In [ ]:
cats = pd.DataFrame()
hit = df[df["bh_llm"]]
if len(hit):
    cats = hit["bh_cat"].value_counts().rename("Calls").to_frame()
    cats["Share of behavioral health calls"] = (cats["Calls"]/len(hit)*100).round(1)
    cats.index.name = "Category"
    cats = cats.reset_index()
    keep(cats, "categories")
    show(cats)
    o = cats.head(12).sort_values("Calls")
    fig, ax = plt.subplots(figsize=(10, max(3, 0.36*len(o)+1)))
    ax.barh(range(len(o)), o["Calls"], color=TEAL)
    ax.set_yticks(range(len(o))); ax.set_yticklabels(o["Category"], fontsize=9)
    ax.set_title("Behavioral health categories across the combined population")
    for i, v in enumerate(o["Calls"]):
        ax.annotate(f"{v:,}", (v, i), xytext=(4,0), textcoords="offset points", va="center", fontsize=8)
    plt.tight_layout(); plt.show()

mkt = pd.DataFrame()
if df["market"].notna().any():
    g = df.groupby(df["market"].fillna("(blank)").astype(str))
    mkt = pd.DataFrame({
        "Distinct calls": g.size(),
        "Notes read": g["bh_read"].sum(),
        "Behavioral health anywhere": g["bh_llm"].sum(),
        "Behavioral health as the reason for the call": g.apply(lambda x: int((x["bh_llm"] & (x["bh_role"]=="primary")).sum()))})
    mkt["Share of calls, anywhere"] = (mkt["Behavioral health anywhere"]/mkt["Distinct calls"]*100).round(1)
    mkt["Share of calls, reason for the call"] = (mkt["Behavioral health as the reason for the call"]/mkt["Distinct calls"]*100).round(1)
    mkt = mkt.sort_values("Distinct calls", ascending=False)
    mkt.index.name = "Market"
    mkt = mkt.reset_index()
    keep(mkt, "by_market")
    show(mkt.head(25))

if df["date"].notna().any():
    t = df.dropna(subset=["date"]).copy()
    t["month"] = t["date"].dt.to_period("M").astype(str)
    t["prim"] = t["bh_llm"] & (t["bh_role"]=="primary")
    tr = t.groupby("month").agg(**{"Distinct calls":("bh_llm","size"),
                                   "Behavioral health anywhere":("bh_llm","sum"),
                                   "Behavioral health as the reason for the call":("prim","sum")})
    tr["Share of calls, anywhere"] = (tr["Behavioral health anywhere"]/tr["Distinct calls"]*100).round(1)
    fig, ax = plt.subplots(figsize=(13,4))
    ax.plot(range(len(tr)), tr["Share of calls, anywhere"], marker="o", lw=2, color=TEAL)
    ax.set_xticks(range(len(tr))); ax.set_xticklabels(tr.index, rotation=60, fontsize=8)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title("Behavioral health rate by month across the combined population")
    plt.tight_layout(); plt.show()
    show(tr.reset_index().rename(columns={"month":"Month"}))

## 10. Write the workbook

In [ ]:
info = [("Source", "extracts combined", ", ".join(SOURCE_FILES)),
        ("Source", "rows across all extracts", f"{before:,}"),
        ("Source", "duplicate rows removed", f"{removed:,}"),
        ("Source", "distinct calls", f"{len(df):,}"),
        ("Source", "date range", f"{df['date'].min().date()} to {df['date'].max().date()}"),
        ("Source", "basis for identifying a call", key_basis),
        ("Source", "extract kept on a duplicate", f"the one whose calls run latest ({ends.index[0]})" if PREFER == "newest" else "the first by file name"),
        ("Run", "large language model", LLM_MODEL),
        ("Run", "prompt fingerprint", PROMPT_FINGERPRINT),
        ("Run", "notes read", f"{int(df['bh_read'].sum()):,} of {len(df):,} distinct calls"),
        ("Run", "notes that could not be read", f"{n_fail:,}"),
        ("How the number was produced", "step 1 - combine",
         "Every extract in the data folder is loaded and tagged with the file it came from. Column names are resolved per file."),
        ("How the number was produced", "step 2 - deduplicate",
         "The extracts cover overlapping periods, so the same call appears more than once. Calls are reduced to distinct calls before anything is counted. Where a call appears in more than one extract the row from the extract whose calls run latest is kept, so a later pull takes precedence over an earlier one."),
        ("How the number was produced", "step 3 - read every note",
         "Every note of at least 20 characters is read by the large language model, which records whether behavioral health is present, whether it was the reason for the call or a secondary factor, the category, and a supporting quote. Notes are keyed by their text, so a call appearing in two extracts is read once."),
        ("How the number was produced", "step 4 - count",
         "Behavioral health calls are counted directly from the note read. Nothing is sampled and nothing is extrapolated, so no figure carries a range."),
        ("How to read it", "reason for the call against secondary",
         "Reason for the call means the note records behavioral health as why the patient called. Secondary means it appears on a call logged under another complaint. The coded field records one reason per call, so the reason-for-the-call figure is the one comparable with it."),
        ("How to read it", "what the counts do not cover",
         "The counts remain subject to errors in the note read and to disagreement about what counts as behavioral health."),
        ("Columns - Source Files", "Extract", "The file the rows came from."),
        ("Columns - Source Files", "Distinct calls first seen in this extract", "Calls counted against this file after duplicates were removed. A call present in two extracts is credited to one of them only."),
        ("Columns - Behavioral Health Counts", "Share of distinct calls", "The measure as a percent of distinct calls, not of rows loaded."),
        ("Columns - Agreement", "Share", "For the first block, the percent of calls carrying a code. For the second, the percent of calls the note says were about behavioral health."),
        ("Columns - By Market", "Notes read", "Calls in that market whose note was long enough to read."),
        ("Columns - Categories", "Share of behavioral health calls", "Share of calls where the note mentions behavioral health anywhere.")]
data_info = pd.DataFrame(info, columns=["Section","Item","Detail"])

def stack(parts):
    out = []
    for section, frame in parts:
        if frame is None or not len(frame): continue
        f = frame.copy(); f.insert(0, "Section", section); out.append(f)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

G = lambda n: RESULTS.get(n)
tabs = {
 "Data Info": data_info,
 "Source Files": stack([("Coverage by extract", G("extract_coverage")),
                        ("Precedence on a duplicate", G("extract_precedence")),
                        ("Deduplication", G("deduplication")),
                        ("Distinct calls by extract", G("extract_contribution"))]),
 "Behavioral Health Counts": G("bh_counts"),
 "Agreement": G("agreement"),
 "Categories": G("categories"),
 "By Market": G("by_market")}

for f in os.listdir(OUT_DIR):
    if f.startswith("Nurse_Navigation_BH_Combined_") and f.endswith(".xlsx"):
        os.remove(os.path.join(OUT_DIR, f))

xlsx = os.path.join(OUT_DIR, f"Nurse_Navigation_BH_Combined_{RUN_ID}.xlsx")
with pd.ExcelWriter(xlsx, engine="xlsxwriter") as xl:
    for name, frame in tabs.items():
        if frame is None or not len(frame): continue
        frame.to_excel(xl, sheet_name=name[:31], index=False)
print("written:", xlsx)
print("tabs:", ", ".join(n for n, f in tabs.items() if f is not None and len(f)))